# Classificação de radiografias de tórax — Parte 3: transferência de aprendizagem

Este é o terceiro notebook do projeto. Compara experiências com CNN e com um modelo pré-treinado usado para extração de características e *fine-tuning*. Durante a execução, cria `data_balanced/` a partir de `data/`; uma pasta `data_balanced/` anterior é removida. Execute-o a partir da raiz do projeto, após as Partes 1 e 2.


## 1. Imports

In [ ]:
#!pip install -U tensorflow numpy pandas matplotlib scipy

# --- Bibliotecas de sistema ---
import os
import random

# --- Números e dados ---
import numpy as np
import pandas as pd
import shutil
from PIL import Image, ImageEnhance

# --- Visualização ---
import matplotlib.pyplot as plt

# --- TensorFlow / Keras ---
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers, metrics
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, array_to_img, load_img
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

In [ ]:
import tensorflow as tf
print(tf.__version__)
import numpy as np
print(np.__version__)


## 2. Setup

In [ ]:
# --- Caminhos dos dados ---
train_dir = 'data/train'
test_dir = 'data/test'

# --- Parâmetros principais ---
batch_size = 32
img_size = (256, 256)

# --- Seed definida para reprodutibilidade ---
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

## 2. Data

### 2.1 Data Load

In [ ]:
# --- Carregar conjunto de treino (labels inferidas pelas pastas) ---
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',  
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    color_mode='grayscale',
)

# --- Carregar conjunto de teste (sem baralhar) ---
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False,
    color_mode='grayscale'
)

### 2.2 Data Analysis

- Class Normal: 0
- Class Pneumonia: 1

Observe que existe um desbalanceamento entre as classes em ambos os datasets, treino e teste, para este primeiro trabalho iremos realizar uma redução no numero de exemplos da class majoritaria.

In [ ]:
# Função para contar o número de exemplos por classe num dataset
def count_exemples_classe(dataset):
    counts = {}

    for _, labels in dataset:
        labels_np = labels.numpy()
        for label in labels_np:
            label = int(label)
            if label in counts:
                counts[label] += 1
            else:
                counts[label] = 1
    
    return counts

# Função para calcular diferença entre classes (para verificar desbalanceamento)
def count_difference(counts):

    if not counts:
        return {}
    
    min_count = min(counts.values())
    diff_counts = {cls: count - min_count for cls, count in counts.items()}
    return diff_counts

In [ ]:
# Contar exemplos nas classes de treino e teste
train_count = count_exemples_classe(train_dataset)
test_count = count_exemples_classe(test_dataset)

# Ordenar resultados por classe
train_count = dict(sorted(train_count.items()))
test_count = dict(sorted(test_count.items()))

print("Train dataset class counts:", train_count)
print("Test dataset class counts:", test_count)

In [ ]:
# Calcular diferenças entre classes
train_diff = count_difference(train_count)
test_diff = count_difference(test_count)

print("Train difference:", train_diff)
print("Test difference:", test_diff)

### 2.3 Data Preparation

In [ ]:
# Online
# --- Normalização dos pixels para [0, 1] ---
normalization_layer = tf.keras.layers.Rescaling(1./255)

# Aplicar normalização aos datasets
train_dataset_normalized = train_dataset.map(lambda x, y: (normalization_layer(x), y))

test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))

# --- Data augmentation (apenas para treino) ---
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
])

train_dataset_online = train_dataset_normalized.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)

# --- Definição de pesos das classes para lidar com desbalanceamento ---
class_weights = {0: 1.94, 1: 0.67}

print("Preparação dos dados concluída.")

In [ ]:
# Offline

# --- Limpar pasta balanceada existente ---
if os.path.exists('data_balanced'):
    shutil.rmtree('data_balanced')
    print("Pasta 'data_balanced' existente removida.")

# --- Copiar a pasta inteira ---
shutil.copytree('data', 'data_balanced')
print("Pasta 'data' copiada para 'data_balanced'.")

# --- Caminho da pasta train balanceada ---
train_balanced_dir = os.path.join('data_balanced', 'train')

# --- Subpastas ---
classes = ['NORMAL', 'PNEUMONIA']

# --- Contar imagens em cada classe ---
counts = {cls: len(os.listdir(os.path.join(train_balanced_dir, cls))) for cls in classes}
major_class = max(counts, key=counts.get)
minor_class = min(counts, key=counts.get)
num_to_generate = counts[major_class] - counts[minor_class]

# --- Exibir resultado ---
print(f"A classe minoritária é: {minor_class} ({counts[minor_class]} imagens)")
print(f"Número de imagens a gerar para balancear: {num_to_generate}")

# --- Diretórios da classe minoritária ---
minor_dir = os.path.join(train_balanced_dir, minor_class)

# --- Lista de imagens da classe minoritária ---
images = os.listdir(minor_dir)
generated = 0

# --- Função para gerar variações simples ---
def augment_image(img_path, save_dir, index, prefix='aug', rotations=10, zoom_range=0.1, translation=0.1):
    img = Image.open(img_path).convert('RGB')
    w, h = img.size

    # --- Zoom aleatório (crop + resize) ---
    zx = random.uniform(1 - zoom_range, 1 + zoom_range)
    zy = random.uniform(1 - zoom_range, 1 + zoom_range)
    crop_w, crop_h = int(w / zx), int(h / zy)
    left = random.randint(0, max(0, w - crop_w))
    top = random.randint(0, max(0, h - crop_h))
    img = img.crop((left, top, left + crop_w, top + crop_h))
    img = img.resize((w, h))

    # --- Brilho aleatório (±20%) ---
    enhancer = ImageEnhance.Brightness(img)
    img = enhancer.enhance(random.uniform(0.8, 1.2))

    # --- Contraste aleatório (±20%) ---
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(random.uniform(0.8, 1.2))

    # --- Leve ruído gaussiano (σ ≈ 0.02) ---
    arr = np.array(img).astype(np.float32) / 255.0
    arr += np.random.normal(0, 0.02, arr.shape)
    arr = np.clip(arr, 0, 1)
    img = Image.fromarray((arr * 255).astype(np.uint8))

    # --- Save imagem ---
    base_name = os.path.basename(img_path)
    save_path = os.path.join(save_dir, f"{prefix}_{index}_{base_name}")
    img.save(save_path)


# --- Gerar imagens aleatórias para balancear ---
while generated < num_to_generate:
    img_file = random.choice(images)
    img_path = os.path.join(minor_dir, img_file)
    augment_image(img_path, minor_dir, generated)
    generated += 1

print("Offline augmentation concluído para a classe minoritária.")


In [ ]:
# --- Carregar conjunto de treino (labels inferidas pelas pastas) ---
train_dir_balanced = 'data_balanced/train'

train_dataset_raw_offline = tf.keras.utils.image_dataset_from_directory(
    train_dir_balanced,
    labels='inferred',  
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    color_mode='grayscale',
)

# --- Normalização dos pixels para [0, 1] ---
normalization_layer = tf.keras.layers.Rescaling(1./255)

# Aplicar normalização aos datasets
train_dataset_offline = train_dataset_raw_offline.map(
    lambda x, y: (normalization_layer(x), y)
)

print("Train dataset offline preparado (normalizado, sem augmentation).")

## 3. CNN

In [ ]:
# --- Modelo CNN ---
modelCNN = tf.keras.models.Sequential([
    layers.Conv2D(16, 5, padding='same', activation='relu', input_shape=(img_size[0], img_size[1], 1)),
    layers.MaxPooling2D(pool_size=(2,2), strides=1, padding='same'),

    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),

    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),

    layers.Dropout(0.4),
    layers.Flatten(),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(1, activation="sigmoid")
])


# --- Callbacks ---
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
# --- Compilação do modelo ---
modelCNN.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall'),
        metrics.AUC(name='auc'),
        metrics.TruePositives(name='tp'),
        metrics.TrueNegatives(name='tn'),
        metrics.FalsePositives(name='fp'),
        metrics.FalseNegatives(name='fn')
    ]
)

# --- Treino (ONLINE) ---
history_online = modelCNN.fit(
    train_dataset_online,        
    validation_data=test_dataset,
    epochs=100,
    class_weight=class_weights,  # usa class weights!
    callbacks=[early_stop]
)

# --- Guardar histórico ---
os.makedirs('results', exist_ok=True)
pd.DataFrame(history_online.history).to_csv('results/model_cnn_online.csv', index=False)

print("Treino ONLINE concluído e histórico guardado.")

In [ ]:
# --- Compilação do modelo ---
modelCNN.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall'),
        metrics.AUC(name='auc'),
        metrics.TruePositives(name='tp'),
        metrics.TrueNegatives(name='tn'),
        metrics.FalsePositives(name='fp'),
        metrics.FalseNegatives(name='fn')
    ]
)
 
# --- Treino (OFFLINE) ---
history_offline = modelCNN.fit(
    train_dataset_offline,
    validation_data=test_dataset,
    epochs=100,
    callbacks=[early_stop] 
)

# --- Guardar histórico ---
pd.DataFrame(history_offline.history).to_csv('results/model_cnn_offline.csv', index=False)

print("Treino OFFLINE concluído e histórico guardado.")


## 4. Feature Extractor

In [ ]:
# --- Carregar conjunto de treino (labels inferidas pelas pastas) ---
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',  
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    color_mode='grayscale',
)

# --- Carregar conjunto de teste (sem baralhar) ---
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False,
    color_mode='grayscale'
)

# --- Normalização dos pixels para [0, 1] ---
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_dataset_normalized = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset_normalized = test_dataset.map(lambda x, y: (normalization_layer(x), y))

# --- Data augmentation (apenas para treino) ---
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
])

train_dataset_augmented = train_dataset_normalized.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)


### 4.1 Model

In [ ]:
# --- Base model pré-treinado (MobileNetV2) ---
base_model = MobileNetV2(
    input_shape=(256, 256, 3),  # 1 canal -> 3 canais
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# --- Construir Feature Extractor ---
inputs = tf.keras.Input(shape=(256, 256, 1))
x = layers.Conv2D(3, (3,3), padding='same')(inputs)  # converter grayscale -> RGB

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)

# Cabeça do classificador
x = layers.Dense(256, activation='relu')(x)       # Dense maior
x = layers.BatchNormalization()(x)               # Normalização da saída
x = layers.Dropout(0.4)(x)                       # Dropout maior para regularização
x = layers.Dense(128, activation='relu')(x)     # Segunda camada Dense
x = layers.BatchNormalization()(x)               # BatchNorm opcional aqui também
x = layers.Dropout(0.3)(x)                       # Dropout menor
outputs = layers.Dense(1, activation='sigmoid')(x)

model_feature_extractor = models.Model(inputs, outputs)

# --- Compilar com otimizador ajustado ---
model_feature_extractor.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),  # learning rate menor
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall'),
        metrics.AUC(name='auc'),
        metrics.TruePositives(name='tp'),
        metrics.TrueNegatives(name='tn'),
        metrics.FalsePositives(name='fp'),
        metrics.FalseNegatives(name='fn')
    ]
)

# --- Callbacks ---
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# --- Treinar Feature Extractor ---
class_weights = {0: 1.94, 1: 0.67}
history_feature_extractor = model_feature_extractor.fit(
    train_dataset_augmented,      # dataset com data augmentation
    validation_data=test_dataset_normalized,
    epochs=30,                     # mais épocas, EarlyStopping controla
    callbacks=[early_stop],
    class_weight=class_weights
)

# --- Guardar histórico ---
os.makedirs('results', exist_ok=True)
pd.DataFrame(history_feature_extractor.history).to_csv(
    'results/model_feature_extractor.csv', index=False
)

print("Treino do Feature Extractor concluído e histórico guardado.")

## 5. Fine-Tuning 

In [ ]:
# --- Carregar conjunto de treino (labels inferidas pelas pastas) ---
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',  
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    color_mode='grayscale',
)

# --- Carregar conjunto de teste (sem baralhar) ---
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False,
    color_mode='grayscale'
)

# --- Normalização ---
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_dataset_normalized = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset_normalized  = test_dataset.map(lambda x, y: (normalization_layer(x), y))

# --- Data augmentation aplicado ao treino ---
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
])
train_dataset_augmented = train_dataset_normalized.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)

### 5.1

In [ ]:
# --- Base model pré-treinado (MobileNetV2) ---
base_model = MobileNetV2(
    input_shape=(256, 256, 3),
    include_top=False,
    weights='imagenet'
)

# --- Descongelar apenas as últimas 20 camadas ---
for layer in base_model.layers[:-20]:
    layer.trainable = False
for layer in base_model.layers[-20:]:
    layer.trainable = True

# --- Construção do modelo Fine-Tuning ---
inputs = tf.keras.Input(shape=(256, 256, 1))
x = layers.Conv2D(3, (3,3), padding='same')(inputs)

x = base_model(x, training=True)
x = layers.GlobalAveragePooling2D()(x)

# --- Cabeça mais forte com BatchNorm ---
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation='sigmoid')(x)

model_finetune = models.Model(inputs, outputs)

# --- Compilação ---
model_finetune.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # LR MUITO baixo para FT
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall'),
        metrics.AUC(name='auc'),
        metrics.TruePositives(name='tp'),
        metrics.TrueNegatives(name='tn'),
        metrics.FalsePositives(name='fp'),
        metrics.FalseNegatives(name='fn'),
    ]
)

# --- Callbacks ---
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# --- Pesos das classes ---
class_weights = {0: 1.94, 1: 0.67}

# --- Treinar Fine-Tuning ---
history_finetune = model_finetune.fit(
    train_dataset_augmented,
    validation_data=test_dataset_normalized,
    epochs=30,
    callbacks=[early_stop],
    class_weight=class_weights
)

# --- Guardar histórico ---
os.makedirs('results', exist_ok=True)
pd.DataFrame(history_finetune.history).to_csv(
    'results/model_finetune.csv', index=False
)

print("Treino Fine-Tuning concluído e histórico guardado.")

## Resultados

In [ ]:
def ultima_epoch(pasta_results):

    # Lista todos os arquivos CSV na pasta
    arquivos = [f for f in os.listdir(pasta_results) if f.endswith('.csv')]
    
    tabela_final = pd.DataFrame()
    
    for arquivo in arquivos:
        caminho_csv = os.path.join(pasta_results, arquivo)
        df = pd.read_csv(caminho_csv)
        
        # Última epoch
        ultima_epoch = df.iloc[-1]
        
        # Lista de todas as métricas existentes no CSV
        todas_metricas = [
            'accuracy','auc','fn','fp','loss','precision','recall','tn','tp',
            'val_accuracy','val_auc','val_fn','val_fp','val_loss','val_precision','val_recall','val_tn','val_tp'
        ]
        
        # Criar série só com métricas existentes
        serie_modelo = pd.Series(
            {m: round(ultima_epoch[m], 4) for m in todas_metricas if m in ultima_epoch},
            name=os.path.splitext(arquivo)[0]  # nome do modelo
        )
        
        # Concatenar ao DataFrame final
        tabela_final = pd.concat([tabela_final, serie_modelo], axis=1)
    
    # Mostrar a tabela
    print("Valores da Última Epoch de todos os modelos:")
    display(tabela_final)
    
    return tabela_final

tabela_modelos = ultima_epoch('results')

In [ ]:
def analisar_modelo(csv_path):
    # Ler CSV
    df = pd.read_csv(csv_path)
    
    # Lista de métricas principais
    metrics = ['accuracy', 'loss', 'precision', 'recall', 'auc']
    
    # Criar gráficos separadamente
    for metric in metrics:
        val_metric = f'val_{metric}'
        
        if metric in df.columns and val_metric in df.columns:
            plt.figure(figsize=(8, 5))
            plt.plot(df[metric], label=f'Train {metric}', marker='o')
            plt.plot(df[val_metric], label=f'Validation {metric}', marker='s')
            plt.title(f'{metric.upper()} por Epoch')
            plt.xlabel('Epoch')
            plt.ylabel(metric.capitalize())
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6)

            # Limitar eixo Y
            if metric == 'loss':
                plt.ylim(0, max(df[metric].max(), df[val_metric].max()) * 1.1)
            else:
                plt.ylim(0, 1.05)

            plt.tight_layout()
            plt.show()
    
    # Última epoch
    ultima_epoch = df.iloc[-1]

    # Lista de todas as métricas possíveis
    todas_metricas = [
        'accuracy','auc','fn','fp','loss','precision','recall','tn','tp',
        'val_accuracy','val_auc','val_fn','val_fp','val_loss','val_precision','val_recall','val_tn','val_tp'
    ]

    # Criar lista de tuplas (métrica, valor) apenas se a métrica existir no CSV
    tabela_valores = [
        (m, round(ultima_epoch[m], 4)) 
        for m in todas_metricas if m in ultima_epoch
    ]

    # Converter para DataFrame
    ultima_epoch_df = pd.DataFrame(tabela_valores, columns=['Métrica', 'Valor'])

    print(f"\nValores da Última Epoch ({csv_path}):")
    display(ultima_epoch_df)


In [ ]:
analisar_modelo('results/model_cnn_offline.csv')

### Interpretação

Os gráficos acima permitem comparar os históricos da CNN com aumento de dados *online* e *offline* nesta execução. Diferenças de treino e validação devem ser avaliadas a partir dos CSV gerados, sem assumir que uma abordagem generaliza melhor em outros dados. O conjunto de teste também é usado como validação durante o treino, pelo que não há uma avaliação final independente. Estes modelos não foram validados para uso clínico.
